# Agent evaluation (Phases 9–10): repair agent vs agent baselines (GPU)

Runs agent systems over the R / C / T scenarios (`eval/agent_scenarios/rct_v2.jsonl`, built from pool v3: all four datasets) and the E / S scenarios (`eval/agent_scenarios/{E,S}/`) with one
real model (same model, limits and budget for every system), and judges each final output against the reference.
Scoring runs on the laptop: `python eval/agent_metrics.py results/agent_v1`.

**Step 1 (`STAGE = "dev"`):** full agent on the dev scenarios only (14 R + 20 C + 6 T + 3 E + 3 S), to check the agent prompt with the real model
(README Phase 9, task 5). **Step 2 (`STAGE = "test"`):** all systems on the test scenarios.
Everything is resumable and cached; outputs per system go to `agent_<system>.zip`.

Settings: GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. *Save Version → Save & Run All*.

**GPU time (estimate):** about 15 s per agent step for the 7B in 4-bit, so ~3 h per system on test (83 R + 24 T + 15 E/S; C is nearly free with the oracle verifier). Run one system per session if needed: it resumes.

In [ ]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/<user>/<repo>.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"; LOAD_IN_4BIT = True     # one model for every agent system
STAGE = "dev"                 # "dev" first (prompt check), then "test"
VERIFIER = "oracle"           # oracle | parsed:<path to parses.jsonl> | conventional
# (system name, mode, extra flags). Main systems first; ablations are optional (R scenarios only, to save GPU time).
SYSTEMS = [
    ("agent_full",     "full",     []),
    ("agent_feedback", "feedback", []),
    ("agent_naive",    "naive",    []),
]
ABLATIONS = [
    ("agent_full_no_memory",    "full", ["--no-memory"]),
    ("agent_full_coarse",       "full", ["--coarse-report"]),
    ("agent_full_no_contract",  "full", ["--hide-contract"]),
    ("agent_full_no_escalate",  "full", ["--no-escalate"]),
]
RUN_ABLATIONS = False
if STAGE == "dev":
    SYSTEMS, RUN_ABLATIONS = SYSTEMS[:1], False
OUT = "/kaggle/working/results/agent_v1"
CACHE = "/kaggle/working/llm_cache"
print(STAGE, [s[0] for s in SYSTEMS], "ablations" if RUN_ABLATIONS else "")

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
NEEDED = ["eval/runner_agent.py", "src/igt/agents/repair_agent.py", "eval/agent_scenarios/rct_v2.jsonl",
          "eval/pool_v3/pool_v3.jsonl", "src/igt/llm/hf.py"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox + bitsandbytes ----------------
# The sandbox and verifier need pandas 2.3.3 / numpy 2.1.3 (pickles, dtype rules). Install them into a separate folder
# and use it only for this repo's processes, so Kaggle's torch stack stays untouched.
subprocess.run(["pip", "install", "-q", "--target", "/kaggle/working/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
if LOAD_IN_4BIT:
    subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
ENV = dict(os.environ, PYTHONPATH="/kaggle/working/evalpkgs")
p = subprocess.run(["python", "-c", "import pandas, numpy, torch; print(pandas.__version__, numpy.__version__, torch.__version__, torch.cuda.is_available())"],
                   env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])

In [ ]:
# ---------------- Run ----------------
def run(system, mode, flags, categories):
    cmd = ["python", "eval/runner_agent.py", "--system", system, "--mode", mode, "--backend", "hf",
           "--model", MODEL_ID, "--verifier", VERIFIER, "--splits", STAGE, "--categories", *categories,
           "--cache-dir", CACHE, "--out", OUT, *flags] + (["--load-in-4bit"] if LOAD_IN_4BIT else [])
    t0 = time.time()
    p = subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    print(p.stdout[-2500:]); print(p.stderr[-2500:])
    assert p.returncode == 0, system
    print(f"== {system} done in {(time.time() - t0) / 60:.1f} min")

for system, mode, flags in SYSTEMS:
    run(system, mode, flags, ["R", "C", "T", "E", "S"])
if RUN_ABLATIONS:
    for system, mode, flags in ABLATIONS:
        run(system, mode, flags, ["R"])

In [ ]:
# ---------------- Quick summary and zip ----------------
p = subprocess.run(["python", "eval/agent_metrics.py", OUT, STAGE], env=ENV, capture_output=True, text=True)
print(p.stdout[-4000:], p.stderr[-1500:])
for d in sorted(os.listdir(OUT)):
    if os.path.isdir(f"{OUT}/{d}"):
        open(f"{OUT}/{d}/git_commit.txt", "w").write(COMMIT)
        print("saved:", shutil.make_archive(f"/kaggle/working/agent_{d}_{STAGE}", "zip", f"{OUT}/{d}"))
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)